In [69]:
from os import urandom

from bitcointx import select_chain_params
from bitcointx.core import b2x, x, lx, b2lx, coins_to_satoshi, CMutableTxIn, CMutableTxOut, COutPoint, CMutableTransaction, Hash160
from bitcointx.wallet import CKeyBase, CCoinExtKey, CCoinExtPubKey, P2WPKHCoinAddress, P2WSHCoinAddress
from bitcointx.core.script import CScript, OP_0, OP_2, OP_CHECKMULTISIG, OP_CHECKSEQUENCEVERIFY, OP_DROP

select_chain_params('bitcoin/regtest')

(<BitcoinRegtestParams: 'bitcoin/regtest'>,
 <BitcoinRegtestParams: 'bitcoin/regtest'>)

In [44]:
# Lightning channels are funded with a P2WSH transaction

# We first need 2 wallets and 2 key pairs
alice = CCoinExtKey.from_seed(urandom(32))
bob = CCoinExtKey.from_seed(urandom(32))

print(f"Alice: {alice}")
print(f"Bob: {bob}")

CURRENT_INDEX = 0
PATH = "m/0'/"

def get_path():
    return PATH + str(CURRENT_INDEX)

alice_funding = alice.derive_path(get_path())
bob_funding = bob.derive_path(get_path())
print(alice_funding.pub)
print(bob_funding.pub)

Alice: tprv8ZgxMBicQKsPee2Da4yeHff6ovQqSCfJAVkpDfLyFjD1T1rmPJofmgMiVagzojvyJq2i1AVfWkHTgu2oVu9ayd8xfp7JCuUjfJ8Dw6YWkTo
Bob: tprv8ZgxMBicQKsPeKWbDiC3n8imENKhRTFcDBbmLrqUkWEqknLjXUbJmEny9v4QB4ZB47mnw9q3xyX28kHDLG4smV324v3A6c5okp1zFACFmmr
CPubKey(x('03f9e0a197843d4f1a8ca6f4b2c5b9c17f2b3a233ab7c5879962aa4362077044eb'))
CPubKey(x('0239e1b771ea9ad5ca0bd36be1e020303186528c0be459525d7d40b564933e6b0b'))


In [45]:
# Alice and Bob exchange pubkeys to create a new multisig address that they both control

multisig_script = CScript([OP_2, bytes(alice_funding.pub), bytes(bob_funding.pub), OP_2, OP_CHECKMULTISIG])
common_output = multisig_script.to_p2wsh_scriptPubKey()
print(f"scriptPubkey: {b2x(common_output)}")
print(f"P2WSH address: {P2WSHCoinAddress.from_scriptPubKey(common_output)}")

scriptPubkey: 0020a24ab5c26a71eb2fde5c06a04416ed73035c05d5ded4ceb90bcf9be20b00561a
P2WSH address: bcrt1q5f9ttsn2w84jlhjuq6syg9hdwvp4cpw4mm2vawgte7d7yzcq2cdq6x5697


In [95]:
# They create a funding transaction

# Here's the UTXO we spend
txid = lx('e549a4b2d00c1cfcf3256c70d0caf5a2a775b3f7f5257fcc7221c5e7453003f2')
vout = 0
all_amount = 50 #bitcoins

# We first create an input
txin = CMutableTxIn(COutPoint(txid, vout))

# We then create an output shared by Alice and Bob
txout = CMutableTxOut(coins_to_satoshi(all_amount), common_output)

funding_tx = CMutableTransaction([txin], [txout])
funding_txid = b2lx(funding_tx.GetTxid())

print(f"Funding transaction id is: {funding_txid}")


Funding transaction id is: 6b060d3008f8eff6ea04bc407da867851b8d4ba55f5c86b5e459e6f55877118d


In [96]:
# They create a first commitment transaction

# We spend the funding transaction
txid = lx(funding_txid)
vout = 0

commitment_txin = CMutableTxIn(COutPoint(txid, vout))

# Let's say that Alice is the funder and Bob the fundee, and that Alice pushes 1 btc to Bob at channel creation
to_alice_amt = 49
to_bob_amt = all_amount - to_alice_amt

# We're gonna need to derive new addresses, so let's increment our index to derive new ones
CURRENT_INDEX = 1
alice_commit = alice.derive_path(get_path())
bob_commit = bob.derive_path(get_path())

# We need 2 versions of the same transaction
# First an "Alice" version, that pays to Bob and locks Alice funds for some time
bob_now_out = CMutableTxOut(coins_to_satoshi(to_bob_amt), CScript([OP_0, Hash160(bob_commit.pub)]))

# We define a delay we must wait before claiming our money back
to_self_delay = b'\x90' # 144 in decimal

# Alice now locks her own money so that she can't claim it before `to_self_delay` is elapsed
alice_delayed = CScript([to_self_delay, OP_CHECKSEQUENCEVERIFY, OP_DROP, alice_commit.pub])
alice_later_out = CMutableTxOut(coins_to_satoshi(to_alice_amt), CScript(alice_delayed))

alice_commit_tx = CMutableTransaction([commitment_txin], [bob_now_out, alice_later_out])
print(f"alice commitment tx is: \n\n{alice_commit_tx}\n")

# then a "Bob" version, that pays Alice immediately and locks Bob funds for some more time
alice_now_out = CMutableTxOut(coins_to_satoshi(to_alice_amt), CScript([OP_0, Hash160(alice_commit.pub)]))

bob_delayed = CScript([to_self_delay, OP_CHECKSEQUENCEVERIFY, OP_DROP, bob_commit.pub])
bob_later_out = CMutableTxOut(coins_to_satoshi(to_bob_amt), CScript(bob_delayed))

bob_commit_tx = CMutableTransaction([commitment_txin], [alice_now_out, bob_later_out])
print(f"bob commitment tx is: \n\n{bob_commit_tx}\n")


alice commitment tx is: 

CBitcoinMutableTransaction([CBitcoinMutableTxIn(CBitcoinMutableOutPoint(lx('6b060d3008f8eff6ea04bc407da867851b8d4ba55f5c86b5e459e6f55877118d'), 0), CBitcoinScript([]), 0xffffffff)], [CBitcoinMutableTxOut(1.0*COIN, CBitcoinScript([0, x('d5217897a36ffe91c594d6d0733fc60907b4f55b')])), CBitcoinMutableTxOut(49.0*COIN, CBitcoinScript([x('90'), OP_CHECKSEQUENCEVERIFY, OP_DROP, x('02d028acf91356330714174b5efbce7709f68be7d131b63606bb49406a2a0fb931')]))], 0, 2, CBitcoinMutableTxWitness([CBitcoinMutableTxInWitness(CScriptWitness([]))]))

bob commitment tx is: 

CBitcoinMutableTransaction([CBitcoinMutableTxIn(CBitcoinMutableOutPoint(lx('6b060d3008f8eff6ea04bc407da867851b8d4ba55f5c86b5e459e6f55877118d'), 0), CBitcoinScript([]), 0xffffffff)], [CBitcoinMutableTxOut(49.0*COIN, CBitcoinScript([0, x('48679a4599a558cd59f3496aa2112096d21587f6')])), CBitcoinMutableTxOut(1.0*COIN, CBitcoinScript([x('90'), OP_CHECKSEQUENCEVERIFY, OP_DROP, x('0274ed11ebb3c802075fab14744739bc0f30e1b53

In [117]:
from bitcointx.core.script import SignatureHash, SIGHASH_ALL, CScriptWitness
from bitcointx.core.scripteval import VerifyScript
from bitcointx.core import CTxInWitness
# Alice and Bob now must sign the commitment transactions

# we first compute the sighash of alice commitment transaction
alice_sighash = SignatureHash(multisig_script, alice_commit_tx, 0, SIGHASH_ALL)

# then alice and bob can sign
alice_sig = alice_funding.priv.sign(alice_sighash) + bytes([SIGHASH_ALL])

bob_sig = bob_funding.priv.sign(alice_sighash) + bytes([SIGHASH_ALL])

# and we can populate the witness of the commitment transaction
scriptWitness = CScriptWitness([alice_sig, bob_sig, multisig_script])
alice_commit_tx.witness = CTxInWitness(scriptWitness)

VerifyScript(CScript([]), common_output, alice_commit_tx, 0, None, 50, alice_commit_tx.witness)

# Now that all is set, Alice can sign and broadcast the funding transaction

AttributeError: 'CBitcoinTxInWitness' object has no attribute 'stack'